# D1 — Basket Trader Hedge Monitor

I built this notebook to simulate the full life cycle of a basket equity trade: from generating synthetic price data through to hedging and PnL attribution. The two strategies I compare are (1) a direct constituent replication hedge and (2) a beta-adjusted single-futures hedge, which is closer to what a real trading desk would use when it can't hold every stock outright.

The project covers three main themes I wanted to practise: vectorised NumPy/Pandas operations for return calculations, object-oriented design using abstract base classes and composition, and realistic PnL accounting for both trade and hedge legs.

## Part 1 — Simulated Price Data

I start by generating 60 business days of synthetic log-normal prices for four large-cap US equities (AAPL, MSFT, NVDA, AMZN). Using `np.random.normal` on log-returns and then `np.cumsum` + `np.exp` is the standard GBM discretisation. I chose per-asset volatilities that roughly reflect historical magnitudes — NVDA is the most volatile at 3% daily, AMZN at 2%, and the others lower. Starting prices are approximate recent levels to keep the numbers realistic.

In [18]:
import pandas as pd 
import numpy as np

### Simulating constituent prices

I simulate each ticker independently with its own daily vol (`sigma`) while sharing a common drift (`mu = 0.0002`). Cumulative-summing the log-returns before exponentiating gives me a path that can never go negative — the key property I need from a price model. I wrap the result in a `pd.DataFrame` with a `DatetimeIndex` from `pd.bdate_range` so weekends are automatically excluded.

### Portfolio weights

I define the basket weights as a `pd.Series` keyed by ticker. The weights sum to 1 and represent value weights, similar to how an index is constructed. Keeping them in a Series rather than a plain dict makes the later `.mul(weights, axis=1)` operations label-aligned and concise.

In [19]:
tickers = ["AAPL","MSFT","NVDA","AMZN"]
n_days = 60

dates = pd.bdate_range("2025-01-01", periods=n_days, freq='B')

S0 = np.array([190, 420, 650, 170], dtype=float)    # Initial price

mu = 0.0002  # daily log-return mean
sigma = np.array([0.015, 0.014, 0.030, 0.020])  # per-asset daily vol

rets = np.random.normal(loc=mu, scale=sigma, size=(n_days, len(tickers)))

prices = pd.DataFrame(
                S0 * np.exp(np.cumsum(rets, axis=0)),
                index=dates, 
                columns=tickers
                )

prices.head()

,AAPL,MSFT,NVDA,AMZN
2025-01-01,188.130744,417.974726,634.462367,164.263654
2025-01-02,188.669830,415.713381,604.300762,165.824240
2025-01-03,186.156754,416.099024,617.787620,166.285816
2025-01-06,189.403582,409.049309,625.414725,164.056665
2025-01-07,186.983081,405.828969,619.720403,164.273906


In [20]:
weights = pd.Series({"AAPL":0.30,"MSFT":0.25,"NVDA":0.25,"AMZN":0.20})
weights

AAPL    0.30
MSFT    0.25
NVDA    0.25
AMZN    0.20
dtype: float64

## Part 2 — Basket Return & Futures Proxy

The basket daily return is the weighted average of constituent simple returns. I chain `.pct_change()` → `.mul(weights)` → `.sum(axis=1)` to get a single return series, then compound it into an index level starting at 100.

For the futures proxy I add a small amount of i.i.d. Gaussian noise (5 bps sigma) to each day's basket return before compounding. This simulates the basis between the basket and a tradeable futures contract — the futures price will track the basket closely but not perfectly, which is the whole point of the beta-hedge section later.

In [21]:
# daily simple returns per ticker
asset_rets = prices.pct_change().fillna(0)

# basket daily return = weighted sum of asset returns
basket_ret = asset_rets.mul(weights, axis=1).sum(axis=1)

# basket index level starting at 100
basket_level = 100 * (1 + basket_ret).cumprod() # Series with a DatetimeIndex

# add noise to returns
np.random.seed(0)

noise_sigma = 0.0005  # 5 bps daily
fut_ret = basket_ret + np.random.normal(0.0, noise_sigma, size=len(basket_ret))

# a proxy that behaves like a tradable future (return-based)
fut_prices = 100 * (1 + pd.Series(fut_ret, index=basket_ret.index)).cumprod()
fut_prices.name = "fut_prices"
fut_prices

2025-01-01    100.088203
2025-01-02     99.059551
2025-01-03     99.343013
2025-01-06     99.593617
2025-01-07     98.908446
2025-01-08     98.428854
2025-01-09    100.639265
2025-01-10    100.740610
2025-01-13    101.754489
2025-01-14    102.706076
2025-01-15    102.578709
2025-01-16    102.223676
2025-01-17    105.562893
2025-01-20    106.029509
2025-01-21    107.359843
2025-01-22    107.990216
2025-01-23    108.163842
2025-01-24    108.275135
2025-01-27    107.782296
2025-01-28    107.816136
2025-01-29    107.561606
2025-01-30    108.910436
2025-01-31    108.868945
2025-02-03    107.806898
2025-02-04    106.997433
2025-02-05    108.245176
2025-02-06    106.540708
2025-02-07    108.127835
2025-02-10    107.877570
2025-02-11    106.972914
2025-02-12    104.880402
2025-02-13    103.803339
2025-02-14    103.124964
2025-02-17    102.944305
2025-02-18    105.888972
2025-02-19    106.821859
2025-02-20    106.287737
2025-02-21    107.341617
2025-02-24    106.624561
2025-02-25    108.615228


### Verifying the basket return formula

Before encapsulating the logic in a class I verify the calculation inline as a quick sanity check. Chaining `.pct_change().fillna(0).head().mul(weights).sum(axis=1)` should reproduce the first few rows of `basket_ret` exactly.

### `Basket` class

I encapsulate the return and level logic in a `Basket` class so the same computation is reusable throughout the notebook without copy-pasting. The class only stores `weights`; all methods take `prices` as an argument, which keeps the object stateless with respect to market data — an important design choice for back-testing where I'll pass different price DataFrames at different points.

In [22]:
prices.pct_change().fillna(0).head().mul(weights).sum(axis=1)

2025-01-01    0.000000
2025-01-02   -0.010478
2025-01-03    0.002372
2025-01-06    0.001402
2025-01-07   -0.007813
Freq: B, dtype: float64

In [23]:
class Basket:
    def __init__(self, weights):
        self.weights = weights 

    def compute_returns(self, prices):
        return prices.pct_change().fillna(0)
    
    def compute_basket_return(self, prices):
        returns = self.compute_returns(prices)
        return returns.mul(self.weights, axis=1).sum(axis=1)    # Weighted average return
    
    def compute_level(self, prices, base=100):
        basket_ret = self.compute_basket_return(prices)
        return base * (1 + basket_ret).cumprod()
    
basket = Basket(weights)
basket_ret = basket.compute_basket_return(prices)
basket_level = basket.compute_level(prices, base=100)

## Part 3 — Trade Object

I model a basket trade as a `Trade` with three attributes: `entry_date`, `notional` (USD), and `side` (+1 = long, -1 = short). Separating the trade from the basket means I can mix and match different weight schemes with the same trade parameters without any code duplication.

### `build_constituent_shares`

This method converts notional into individual share counts at entry-date prices. The formula is: `shares_i = side * notional * w_i / S_i(entry_date)`. Fixing shares at entry and holding them constant through the simulation gives a clean definition of PnL: daily delta_S times shares held, summed across tickers.

In [24]:
class Trade:
    def __init__(self, entry_date, notional, side):
        self.entry_date = entry_date
        self.notional = notional
        self.side = side

    def build_constituent_shares(self, prices, basket: Basket):
        entry_prices = prices.loc[self.entry_date]
        w = basket.weights.reindex(prices.columns)  # Just to double confirm correct index
        
        shares = self.side * self.notional * w / entry_prices
        shares.name = "shares"
        return shares

trade = Trade(entry_date=prices.index[0], notional=10_000_000, side=+1)
trade_shares = trade.build_constituent_shares(prices, basket)
trade_shares

AAPL    15946.357005
MSFT     5981.222891
NVDA     3940.344029
AMZN    12175.547955
Name: shares, dtype: float64

## Part 4 — Hedge Strategy (Abstract Base Class)

I use Python's `abc.ABC` to define `HedgeStrategy` as an interface. Any concrete strategy must implement `build_hedge(prices, basket, trade)`. This design lets `Backtester` work against any strategy without knowing its internals — a classic strategy pattern that becomes useful when I swap between the replication and futures hedges further down.

### `ReplicationHedge`

The simplest possible hedge: hold exactly the negative of the trade's constituent shares. By construction this produces zero net PnL every day because every price move in the trade leg is cancelled identically in the hedge leg. I use this as a ground-truth baseline to verify that my `Backtester` PnL accounting is correct before introducing the noisy futures hedge.

The replication hedge shares are the negation of `trade_shares`: `hedge_shares = -trade_shares`. The opposite sign offsets the trade's directional exposure, and because the same underlying prices drive both legs, the net PnL is identically zero — no approximation involved.

In [25]:
from abc import ABC, abstractmethod

class HedgeStrategy(ABC):
    """
    Strategy interface.
    Any hedge strategy must implement build_hedge(...).
    """

    @abstractmethod
    def build_hedge(self, prices, basket, trade):
        """
        Return a hedge object / positions that can be valued over time.
        
        Parameters
        ----------
        prices : pd.DataFrame
            Wide prices (dates x tickers).
        basket : Basket
            Basket instance (has weights, methods, etc.)
        trade : Trade
            Trade instance (entry_date, notional, side, shares, etc.)

        Returns
        -------
        hedge : object
            We'll define later (could be Series of futures shares, dict of positions, etc.)
        """
        raise NotImplementedError
    
class ReplicationHedge(HedgeStrategy):  # Uses HedgeStrategy as a template
    def build_hedge(self, prices, basket, trade) -> pd.Series:
        """
        Replication hedge = - trade_shares (same tickers, opposite shares)
        Returns a pd.Series indexed by ticker.
        """
        trade_shares = trade.build_constituent_shares(prices, basket)  # Series
        hedge_shares = -trade_shares
        hedge_shares.name = "hedge_shares"
        return hedge_shares


In [26]:
# Usage example
rep_hedge = ReplicationHedge()
hedge_shares = rep_hedge.build_hedge(prices, basket, trade)
hedge_shares

AAPL   -15946.357005
MSFT    -5981.222891
NVDA    -3940.344029
AMZN   -12175.547955
Name: hedge_shares, dtype: float64

## Part 5 — Backtester

`Backtester` is responsible for turning share positions into PnL series. `pnl_from_shares` computes `sum_i shares_i * delta_P_{t,i}` for every date using `.diff()` on the price DataFrame. I `.reindex` shares to the price columns before the dot product to guard against any column-order misalignment.

### `run_replication`

This method wires `Trade`, `Basket`, and `ReplicationHedge` together and returns a three-column DataFrame: `pnl_trade`, `pnl_hedge`, `pnl_net`. I verify that `pnl_net` cumulates to (near) zero across all 60 days — any deviation would indicate a bug in the accounting.

In [27]:
class Backtester:
    def pnl_from_shares(self, prices: pd.DataFrame, shares: pd.Series) -> pd.Series:
        """
        prices: DataFrame (dates x tickers)
        shares: Series (tickers,)  -> constant shares held through time
        returns: Series (dates,) daily PnL
        """
        # 1) daily price changes per ticker
        dP = prices.diff().fillna(0)

        # 2) align shares to columns (super important)
        shares = shares.reindex(prices.columns)

        # 3) daily pnl = sum_i shares_i * dP_{t,i}
        pnl = dP.mul(shares, axis=1).sum(axis=1)
        pnl.name = "pnl"
        return pnl

    def run_replication(self, prices: pd.DataFrame, basket, trade, hedge) -> pd.DataFrame:
        """
        Runs: trade + replication hedge and returns results DataFrame.
        """
        # shares for trade and hedge
        trade_shares = trade.build_constituent_shares(prices, basket)      # Series
        # The reason for creating HedgeStrategy template is for 
        # unified method usage here
        hedge_shares = hedge.build_hedge(prices, basket, trade)            # Series

        # pnl series
        pnl_trade = self.pnl_from_shares(prices, trade_shares).rename("pnl_trade")
        pnl_hedge = self.pnl_from_shares(prices, hedge_shares).rename("pnl_hedge")

        # net
        pnl_net = (pnl_trade + pnl_hedge).rename("pnl_net")

        # pack results
        results = pd.concat([pnl_trade, pnl_hedge, pnl_net], axis=1)

        return results

In [28]:
# Usage 
bt = Backtester()
rep_results = bt.run_replication(prices, basket, trade, rep_hedge)
rep_results[["pnl_trade", "pnl_hedge", "pnl_net"]].cumsum().tail()

,pnl_trade,pnl_hedge,pnl_net
2025-03-19,132963.572070,-132963.572070,0.0
2025-03-20,162197.504802,-162197.504802,0.0
2025-03-21,175155.027772,-175155.027772,0.0
2025-03-24,294954.800008,-294954.800008,0.0
2025-03-25,316374.636218,-316374.636218,0.0


## Part 6 — Futures Beta Hedge

Real desks often can't hold all basket constituents individually — they use a single liquid futures contract instead and size it via a beta (sensitivity ratio). I construct a futures proxy by adding noise to the basket returns, then estimate `beta = cov(basket, futures) / var(futures)` over the full sample. The hedge notional is then `beta * trade_notional`.

In [29]:
# --- basket returns and level ---
asset_rets = prices.pct_change().fillna(0)
basket_ret = asset_rets.mul(basket.weights.reindex(prices.columns), axis=1).sum(axis=1)
basket_level = 100 * (1 + basket_ret).cumprod()

# --- futures proxy ---
np.random.seed(0)

noise_sigma = 0.001  # 10 bps daily noise; try 0.0005 to 0.002
fut_ret = basket_ret + np.random.normal(0.0, noise_sigma, size=len(basket_ret))

fut_prices = 100 * (1 + pd.Series(fut_ret, index=prices.index)).cumprod()
fut_prices.name = "fut_prices"

### `FuturesBetaHedge`

The class wraps the beta estimation and returns both `hedge_notional` and `beta` so the caller can inspect the sizing. I keep two approaches commented in the code: Approach 1 uses only a rolling lookback window ending at `entry_date` (more realistic), while Approach 2 uses the full sample (simpler for illustration). I default to Approach 2 here to keep the focus on structure rather than look-ahead bias.

Beta is estimated as: `beta = Cov(basket_returns, futures_returns) / Var(futures_returns)`. This is the same formula as a linear regression slope coefficient. Hedging with `beta * notional` in futures minimises the residual variance of the combined position — it is not a perfect hedge because of basis noise, but it captures the dominant systematic exposure.

In [30]:
class FuturesBetaHedge(HedgeStrategy):
    def build_hedge(self, prices, basket, trade, fut_prices, lookback=20):
        """
        Returns (hedge_notional, beta)
        hedge_notional is the magnitude to hedge (positive).
        beta estimated using lookback window ending at entry_date.
        """
        # basket returns
        asset_rets = prices.pct_change().fillna(0)
        basket_ret = asset_rets.mul(basket.weights.reindex(prices.columns), axis=1).sum(axis=1)
        
        # futures returns
        fut_ret = fut_prices.pct_change().fillna(0)

        """
        # Approach 1
        # use lookback up to entry_date (inclusive)
        end = trade.entry_date
        window = basket_ret.loc[:end].tail(lookback) 
        fut_window = fut_ret.loc[:end].tail(lookback)
        """
        
        # Approach 2
        # use all data (not realistic but easy) for lookback
        window = basket_ret
        fut_window = fut_ret

        # beta = cov(basket, fut) / var(fut)
        var_f = fut_window.var()
        if var_f == 0:
            beta = 0.0
        else:
            beta = window.cov(fut_window) / var_f

        hedge_notional = float(beta) * trade.notional
        return hedge_notional, float(beta)

In [31]:
# Usage
fut_hedge = FuturesBetaHedge()
hedge_notional, beta = fut_hedge.build_hedge(prices, basket, trade, fut_prices, lookback=20)

print("beta:", beta)
print("hedge_notional:", hedge_notional)

beta: 0.9902994212873564
hedge_notional: 9902994.212873563


### Converting notional to futures units

The number of futures contracts to short is: `units = -side * hedge_notional / fut_entry_price`. I anchor the units at the entry-date futures price so the hedge size is fixed at inception, consistent with how the trade shares were fixed. The negative sign ensures a long basket trade is offset by a short futures position.

### Extending `Backtester` with `run_futures`

I add `pnl_from_futures_units` — a simple scalar-times-delta_F computation — and `run_futures` to wire the futures-hedged back-test end to end. The results DataFrame includes `beta`, `hedge_notional`, and `fut_units` alongside the PnL columns so I can inspect the sizing at a glance.

units = -trade.side * hedge_notional / fut_entry_price

Backtester extension

In [32]:
(prices.diff().fillna(0) * trade_shares).head()
# prices.diff().fillna(0) 

,AAPL,MSFT,NVDA,AMZN
2025-01-01,0.000000,0.000000,0.000000,0.000000
2025-01-02,8596.469471,-13525.611826,-118847.101651,19000.981920
2025-01-03,-40074.411631,2306.616648,53142.859783,5619.939803
2025-01-06,51775.077113,-42165.916122,30053.420075,-27141.129513
2025-01-07,-38598.176441,-19261.568531,-22437.586986,2645.028387


In [33]:
class Backtester:
    def pnl_from_shares(self, prices: pd.DataFrame, shares: pd.Series) -> pd.Series:
        dP = prices.diff().fillna(0)
        shares = shares.reindex(prices.columns)
        pnl = dP.mul(shares, axis=1).sum(axis=1)
        pnl.name = "pnl"
        return pnl

    def pnl_from_futures_units(self, fut_prices: pd.Series, units: float) -> pd.Series:
        dF = fut_prices.diff().fillna(0)
        pnl = units * dF
        pnl.name = "pnl"
        return pnl
    
    def run_replication(self, prices: pd.DataFrame, basket, trade, hedge) -> pd.DataFrame:
        trade_shares = trade.build_constituent_shares(prices, basket)
        hedge_shares = hedge.build_hedge(prices, basket, trade)
        
        pnl_trade = self.pnl_from_shares(prices, trade_shares).rename("pnl_trade")
        pnl_hedge = self.pnl_from_shares(prices, hedge_shares).rename("pnl_hedge")
        pnl_net = (pnl_trade + pnl_hedge).rename("pnl_net")

        return pd.concat([pnl_trade, pnl_hedge, pnl_net], axis=1)

    def run_futures(self, prices, fut_prices, basket, trade, fut_hedge, lookback=20) -> pd.DataFrame:
        # trade shares + pnl
        trade_shares = trade.build_constituent_shares(prices, basket)
        pnl_trade = self.pnl_from_shares(prices, trade_shares).rename("pnl_trade")

        # hedge params
        hedge_notional, beta = fut_hedge.build_hedge(prices, basket, trade, fut_prices, lookback=lookback)
        
        # futures units sized at entry
        fut_entry = fut_prices.loc[trade.entry_date]
        units = (-trade.side) * hedge_notional / fut_entry  # short if long basket

        pnl_hedge = self.pnl_from_futures_units(fut_prices, units).rename("pnl_hedge")
        pnl_net = (pnl_trade + pnl_hedge).rename("pnl_net")

        results = pd.concat([pnl_trade, pnl_hedge, pnl_net], axis=1)
        results["beta"] = beta
        results["hedge_notional"] = hedge_notional
        results["fut_units"] = units
        return results

The replication hedge produces near-zero net PnL as expected.  
The futures beta hedge materially reduces risk but leaves residual tracking error due to imperfect correlation and basis noise.

## Part 7 — Comparing Both Hedges

Running both back-tests side by side confirms the expected behaviour: the replication hedge nets to exactly zero, while the futures beta hedge leaves residual PnL (tracking error) because the futures proxy is not a perfect substitute for the basket. The magnitude of that residual grows with the noise sigma used to construct the proxy — a sensitivity worth experimenting with.

In [34]:
bt = Backtester()

rep_results = bt.run_replication(prices, basket, trade, rep_hedge)
fut_results = bt.run_futures(prices, fut_prices, basket, trade, fut_hedge, lookback=20)

print(rep_results[["pnl_trade","pnl_hedge","pnl_net"]].cumsum().tail())
print(fut_results[["pnl_trade","pnl_hedge","pnl_net"]].cumsum().tail())

                pnl_trade      pnl_hedge  pnl_net
2025-03-19  132963.572070 -132963.572070      0.0
2025-03-20  162197.504802 -162197.504802      0.0
2025-03-21  175155.027772 -175155.027772      0.0
2025-03-24  294954.800008 -294954.800008      0.0
2025-03-25  316374.636218 -316374.636218      0.0
                pnl_trade      pnl_hedge       pnl_net
2025-03-19  132963.572070 -165077.849563 -32114.277493
2025-03-20  162197.504802 -173978.124510 -11780.619708
2025-03-21  175155.027772 -190502.527828 -15347.500056
2025-03-24  294954.800008 -291599.927007   3354.873002
2025-03-25  316374.636218 -300101.968767  16272.667451
